# 三种策略对比: max(0,6) vs max(2,6) vs max(0,2,6)

## 周末报价策略比较

- **策略1 (max(0,6))**: 用 `max(hour_0, hour_6)` 的USDCNH作为周末报价
- **策略2 (max(2,6))**: 用 `max(hour_2, hour_6)` 的USDCNH作为周末报价
- **策略3 (max(0,2,6))**: 用 `max(hour_0, hour_2, hour_6)` 的USDCNH作为周末报价 **[NEW]**

### 交易量
- USD: 60M USD/周
- JPY: 4B JPY/周

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.font_manager import FontProperties
import warnings
warnings.filterwarnings('ignore')

# ======== 中文字体设置 (Windows) ========
font_path = r'C:\Windows\Fonts\msyh.ttc'  # 微软雅黑
if not os.path.exists(font_path):
    font_path = r'C:\Windows\Fonts\simhei.ttf'
if not os.path.exists(font_path):
    font_path = r'C:\Windows\Fonts\simsun.ttc'

chinese_font = FontProperties(fname=font_path, size=12)
chinese_font_title = FontProperties(fname=font_path, size=14)
chinese_font_legend = FontProperties(fname=font_path, size=11)
print(f'Using font: {font_path}')

OUTPUT_DIR = r'c:\Users\tencentren\CodeBuddy\FX_SYSTEM\bmad-quant-system\output'

# 交易量
USD_WEEKLY_VOLUME = 60_000_000      # 6000万 USD/周
JPY_WEEKLY_VOLUME = 4_000_000_000   # 40亿 JPY/周

print('\n交易量设置:')
print(f'  USD: {USD_WEEKLY_VOLUME/1e6:.0f}M USD/周')
print(f'  JPY: {JPY_WEEKLY_VOLUME/1e9:.0f}B JPY/周')

## 1. 加载数据

In [ ]:
# 加载USDCNH和JPYCNH数据
all_files = os.listdir(OUTPUT_DIR)

# USDCNH 1年数据
usdcnh_files = [f for f in all_files if 'USDCNH' in f and '1year' in f and f.endswith('.xlsx') and not f.startswith('~$')]
print(f"Loading USDCNH: {usdcnh_files[0]}")
df_usdcnh = pd.read_excel(os.path.join(OUTPUT_DIR, usdcnh_files[0]))
df_usdcnh['timestamp'] = pd.to_datetime(df_usdcnh['timestamp'])
df_usdcnh.set_index('timestamp', inplace=True)

# JPYCNH 1年数据
jpycnh_files = [f for f in all_files if 'JPYCNH' in f and '1year' in f and f.endswith('.xlsx') and not f.startswith('~$')]
print(f"Loading JPYCNH: {jpycnh_files[0]}")
df_jpycnh = pd.read_excel(os.path.join(OUTPUT_DIR, jpycnh_files[0]))
df_jpycnh['timestamp'] = pd.to_datetime(df_jpycnh['timestamp'])
df_jpycnh.set_index('timestamp', inplace=True)

print(f"\nUSDCNH: {len(df_usdcnh)} rows, {df_usdcnh.index.min()} ~ {df_usdcnh.index.max()}")
print(f"JPYCNH: {len(df_jpycnh)} rows, {df_jpycnh.index.min()} ~ {df_jpycnh.index.max()}")

In [ ]:
# 准备周六数据 (0-6点)
for df in [df_usdcnh, df_jpycnh]:
    df['weekday'] = df.index.dayofweek
    df['date'] = df.index.date
    df['hour'] = df.index.hour
    if 'mid' not in df.columns:
        df['mid'] = (df['bid'] + df['ask']) / 2

usdcnh_sat = df_usdcnh[(df_usdcnh['weekday'] == 5) & (df_usdcnh['hour'] <= 6)].copy()
jpycnh_sat = df_jpycnh[(df_jpycnh['weekday'] == 5) & (df_jpycnh['hour'] <= 6)].copy()

# 共同日期
common_dates = sorted(set(usdcnh_sat['date'].unique()) & set(jpycnh_sat['date'].unique()))
print(f"Common Saturday dates: {len(common_dates)} weeks")
print(f"Date range: {common_dates[0]} ~ {common_dates[-1]}")

## 2. 计算三种策略的PnL

In [ ]:
# 计算每周的策略PnL - 三种策略
weekly_data = []

for sat_date in common_dates:
    usdcnh_day = usdcnh_sat[usdcnh_sat['date'] == sat_date]
    jpycnh_day = jpycnh_sat[jpycnh_sat['date'] == sat_date]
    
    if usdcnh_day.empty or jpycnh_day.empty:
        continue
    
    # 获取各小时USDCNH价格
    usdcnh_h0 = usdcnh_day[usdcnh_day['hour'] == 0]['mid'].mean() if len(usdcnh_day[usdcnh_day['hour'] == 0]) > 0 else np.nan
    usdcnh_h2 = usdcnh_day[usdcnh_day['hour'] == 2]['mid'].mean() if len(usdcnh_day[usdcnh_day['hour'] == 2]) > 0 else np.nan
    usdcnh_h6 = usdcnh_day[usdcnh_day['hour'] == 6]['mid'].mean() if len(usdcnh_day[usdcnh_day['hour'] == 6]) > 0 else np.nan
    
    # 获取JPYCNH价格 (每100日元)
    jpycnh_avg = jpycnh_day['mid'].mean()
    jpycnh_per_jpy = jpycnh_avg / 100
    
    if pd.isna(usdcnh_h0) or pd.isna(usdcnh_h2) or pd.isna(usdcnh_h6):
        continue
    
    # 三种策略的参照物
    ref_max_0_6 = max(usdcnh_h0, usdcnh_h6)              # 策略1: max(0,6)
    ref_max_2_6 = max(usdcnh_h2, usdcnh_h6)              # 策略2: max(2,6)
    ref_max_0_2_6 = max(usdcnh_h0, usdcnh_h2, usdcnh_h6) # 策略3: max(0,2,6)
    
    usdcnh_avg = usdcnh_day['mid'].mean()
    
    # ===== max(0,6) vs max(2,6) =====
    usd_diff_06_vs_26 = (ref_max_0_6 - ref_max_2_6) * USD_WEEKLY_VOLUME
    jpy_diff_06_vs_26 = (jpycnh_per_jpy * (ref_max_0_6 / usdcnh_avg) - jpycnh_per_jpy * (ref_max_2_6 / usdcnh_avg)) * JPY_WEEKLY_VOLUME
    
    # ===== max(0,2,6) vs max(0,6) =====
    usd_diff_026_vs_06 = (ref_max_0_2_6 - ref_max_0_6) * USD_WEEKLY_VOLUME
    jpy_diff_026_vs_06 = (jpycnh_per_jpy * (ref_max_0_2_6 / usdcnh_avg) - jpycnh_per_jpy * (ref_max_0_6 / usdcnh_avg)) * JPY_WEEKLY_VOLUME
    
    # ===== max(0,2,6) vs max(2,6) =====
    usd_diff_026_vs_26 = (ref_max_0_2_6 - ref_max_2_6) * USD_WEEKLY_VOLUME
    jpy_diff_026_vs_26 = (jpycnh_per_jpy * (ref_max_0_2_6 / usdcnh_avg) - jpycnh_per_jpy * (ref_max_2_6 / usdcnh_avg)) * JPY_WEEKLY_VOLUME
    
    weekly_data.append({
        'date': sat_date,
        'usdcnh_h0': usdcnh_h0,
        'usdcnh_h2': usdcnh_h2,
        'usdcnh_h6': usdcnh_h6,
        'ref_max_0_6': ref_max_0_6,
        'ref_max_2_6': ref_max_2_6,
        'ref_max_0_2_6': ref_max_0_2_6,
        # max(0,6) vs max(2,6)
        'usd_diff_06_vs_26': usd_diff_06_vs_26,
        'jpy_diff_06_vs_26': jpy_diff_06_vs_26,
        'total_diff_06_vs_26': usd_diff_06_vs_26 + jpy_diff_06_vs_26,
        # max(0,2,6) vs max(0,6)
        'usd_diff_026_vs_06': usd_diff_026_vs_06,
        'jpy_diff_026_vs_06': jpy_diff_026_vs_06,
        'total_diff_026_vs_06': usd_diff_026_vs_06 + jpy_diff_026_vs_06,
        # max(0,2,6) vs max(2,6)
        'usd_diff_026_vs_26': usd_diff_026_vs_26,
        'jpy_diff_026_vs_26': jpy_diff_026_vs_26,
        'total_diff_026_vs_26': usd_diff_026_vs_26 + jpy_diff_026_vs_26,
    })

df = pd.DataFrame(weekly_data)
print(f"计算完成: {len(df)} 周数据")

In [ ]:
# 计算累计PnL
df['cum_06_vs_26'] = df['total_diff_06_vs_26'].cumsum()
df['cum_026_vs_06'] = df['total_diff_026_vs_06'].cumsum()
df['cum_026_vs_26'] = df['total_diff_026_vs_26'].cumsum()

# USD + JPY 分开累计
df['usd_cum_06_vs_26'] = df['usd_diff_06_vs_26'].cumsum()
df['jpy_cum_06_vs_26'] = df['jpy_diff_06_vs_26'].cumsum()
df['usd_cum_026_vs_06'] = df['usd_diff_026_vs_06'].cumsum()
df['jpy_cum_026_vs_06'] = df['jpy_diff_026_vs_06'].cumsum()
df['usd_cum_026_vs_26'] = df['usd_diff_026_vs_26'].cumsum()
df['jpy_cum_026_vs_26'] = df['jpy_diff_026_vs_26'].cumsum()

print("="*70)
print("累计PnL差异汇总 (美金 + 日元)")
print("="*70)
print(f"max(0,6) vs max(2,6):   {df['cum_06_vs_26'].iloc[-1]:>12,.0f} CNH ({df['cum_06_vs_26'].iloc[-1]/1e6:.2f}M)")
print(f"max(0,2,6) vs max(0,6): {df['cum_026_vs_06'].iloc[-1]:>12,.0f} CNH ({df['cum_026_vs_06'].iloc[-1]/1e6:.2f}M)")
print(f"max(0,2,6) vs max(2,6): {df['cum_026_vs_26'].iloc[-1]:>12,.0f} CNH ({df['cum_026_vs_26'].iloc[-1]/1e6:.2f}M)")

## 3. 绘制三种策略参照价对比

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

dates = pd.to_datetime(df['date'])

ax.plot(dates, df['ref_max_0_6'], 'b-', linewidth=2, marker='o', markersize=5, label='max(0,6)')
ax.plot(dates, df['ref_max_2_6'], 'g-', linewidth=2, marker='s', markersize=5, label='max(2,6)')
ax.plot(dates, df['ref_max_0_2_6'], 'r-', linewidth=2.5, marker='^', markersize=6, label='max(0,2,6)')

ax.set_xlabel('日期', fontproperties=chinese_font)
ax.set_ylabel('USDCNH 参照价', fontproperties=chinese_font)
ax.set_title('三种策略的USDCNH参照价对比', fontproperties=chinese_font_title)
ax.legend(['max(0,6)', 'max(2,6)', 'max(0,2,6)'], loc='upper left', prop=chinese_font_legend)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

## 4. 绘制累计PnL对比 (以max(2,6)为基准)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

dates = pd.to_datetime(df['date'])

ax.plot(dates, df['cum_06_vs_26']/1e6, 'b-', linewidth=2.5, marker='o', markersize=6)
ax.plot(dates, df['cum_026_vs_26']/1e6, 'r-', linewidth=2.5, marker='^', markersize=6)
ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.fill_between(dates, 0, df['cum_026_vs_26']/1e6, where=(df['cum_026_vs_26'] >= 0), color='red', alpha=0.1)

final_06_vs_26 = df['cum_06_vs_26'].iloc[-1]/1e6
final_026_vs_26 = df['cum_026_vs_26'].iloc[-1]/1e6

ax.annotate(f'max(0,6): {final_06_vs_26:.2f}M', xy=(dates.iloc[-1], final_06_vs_26), xytext=(10, 0),
            textcoords='offset points', fontsize=11, color='blue', fontweight='bold', fontproperties=chinese_font)
ax.annotate(f'max(0,2,6): {final_026_vs_26:.2f}M', xy=(dates.iloc[-1], final_026_vs_26), xytext=(10, 5),
            textcoords='offset points', fontsize=11, color='red', fontweight='bold', fontproperties=chinese_font,
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

ax.set_xlabel('日期', fontproperties=chinese_font)
ax.set_ylabel('累计PnL差异 (百万 CNH)', fontproperties=chinese_font)
ax.set_title('累计PnL对比 (相对于 max(2,6) 基准)\n正值=该策略更优', fontproperties=chinese_font_title)
legend_labels = ['max(0,6) - max(2,6)', 'max(0,2,6) - max(2,6)']
ax.legend(legend_labels, loc='upper left', prop=chinese_font_legend)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

## 5. max(0,2,6) vs max(0,6) 分币种对比

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

dates = pd.to_datetime(df['date'])

ax.plot(dates, df['usd_cum_026_vs_06']/1e6, 'b-', linewidth=2.5, marker='o', markersize=5)
ax.plot(dates, df['jpy_cum_026_vs_06']/1e6, 'g-', linewidth=2.5, marker='s', markersize=5)
ax.plot(dates, df['cum_026_vs_06']/1e6, 'r-', linewidth=3, marker='^', markersize=6)
ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.fill_between(dates, 0, df['cum_026_vs_06']/1e6, where=(df['cum_026_vs_06'] >= 0), color='green', alpha=0.1)
ax.fill_between(dates, 0, df['cum_026_vs_06']/1e6, where=(df['cum_026_vs_06'] < 0), color='red', alpha=0.1)

final_usd = df['usd_cum_026_vs_06'].iloc[-1]/1e6
final_jpy = df['jpy_cum_026_vs_06'].iloc[-1]/1e6
final_total = df['cum_026_vs_06'].iloc[-1]/1e6

ax.annotate(f'美金: {final_usd:.2f}M', xy=(dates.iloc[-1], final_usd), xytext=(10, 0),
            textcoords='offset points', fontsize=10, color='blue', fontweight='bold', fontproperties=chinese_font)
ax.annotate(f'日元: {final_jpy:.2f}M', xy=(dates.iloc[-1], final_jpy), xytext=(10, 0),
            textcoords='offset points', fontsize=10, color='green', fontweight='bold', fontproperties=chinese_font)
ax.annotate(f'合计: {final_total:.2f}M', xy=(dates.iloc[-1], final_total), xytext=(10, 5),
            textcoords='offset points', fontsize=11, color='red', fontweight='bold', fontproperties=chinese_font,
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

ax.set_xlabel('日期', fontproperties=chinese_font)
ax.set_ylabel('累计PnL差异 (百万 CNH)', fontproperties=chinese_font)
ax.set_title('max(0,2,6) vs max(0,6) 累计PnL差异\n正值=max(0,2,6)更优', fontproperties=chinese_font_title)
legend_labels = ['美金', '日元', '合计']
ax.legend(legend_labels, loc='upper left', prop=chinese_font_legend)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

## 6. 详细统计汇总

In [ ]:
print("="*70)
print("三种策略对比统计汇总")
print("="*70)

print(f"\n回测期间: {df['date'].iloc[0]} ~ {df['date'].iloc[-1]}")
print(f"总周数: {len(df)}")

print(f"\n交易量:")
print(f"  美金: {USD_WEEKLY_VOLUME/1e6:.0f}M USD/周")
print(f"  日元: {JPY_WEEKLY_VOLUME/1e9:.0f}B JPY/周")

print(f"\n" + "="*70)
print("累计PnL差异 (正值=前者更优)")
print("="*70)

print(f"\n1. max(0,6) vs max(2,6):")
print(f"   美金: {df['usd_cum_06_vs_26'].iloc[-1]:>12,.0f} CNH")
print(f"   日元: {df['jpy_cum_06_vs_26'].iloc[-1]:>12,.0f} CNH")
print(f"   合计: {df['cum_06_vs_26'].iloc[-1]:>12,.0f} CNH ({df['cum_06_vs_26'].iloc[-1]/1e6:.2f}M)")

print(f"\n2. max(0,2,6) vs max(0,6):")
print(f"   美金: {df['usd_cum_026_vs_06'].iloc[-1]:>12,.0f} CNH")
print(f"   日元: {df['jpy_cum_026_vs_06'].iloc[-1]:>12,.0f} CNH")
print(f"   合计: {df['cum_026_vs_06'].iloc[-1]:>12,.0f} CNH ({df['cum_026_vs_06'].iloc[-1]/1e6:.2f}M)")

print(f"\n3. max(0,2,6) vs max(2,6):")
print(f"   美金: {df['usd_cum_026_vs_26'].iloc[-1]:>12,.0f} CNH")
print(f"   日元: {df['jpy_cum_026_vs_26'].iloc[-1]:>12,.0f} CNH")
print(f"   合计: {df['cum_026_vs_26'].iloc[-1]:>12,.0f} CNH ({df['cum_026_vs_26'].iloc[-1]/1e6:.2f}M)")

print(f"\n" + "="*70)
print("结论: max(0,2,6) 策略最优!")
print(f"  相比 max(0,6): 多赚 {df['cum_026_vs_06'].iloc[-1]:,.0f} CNH ({df['cum_026_vs_06'].iloc[-1]/1e6:.2f}M)")
print(f"  相比 max(2,6): 多赚 {df['cum_026_vs_26'].iloc[-1]:,.0f} CNH ({df['cum_026_vs_26'].iloc[-1]/1e6:.2f}M)")
print("="*70)

## 7. 每周明细数据

In [ ]:
# 显示每周明细
display_df = df[['date', 'usdcnh_h0', 'usdcnh_h2', 'usdcnh_h6',
                 'ref_max_0_6', 'ref_max_2_6', 'ref_max_0_2_6',
                 'total_diff_06_vs_26', 'total_diff_026_vs_06', 'total_diff_026_vs_26',
                 'cum_06_vs_26', 'cum_026_vs_06', 'cum_026_vs_26']].copy()

# 格式化数值列
for col in ['total_diff_06_vs_26', 'total_diff_026_vs_06', 'total_diff_026_vs_26',
            'cum_06_vs_26', 'cum_026_vs_06', 'cum_026_vs_26']:
    display_df[col] = display_df[col].apply(lambda x: f"{x:,.0f}")

for col in ['usdcnh_h0', 'usdcnh_h2', 'usdcnh_h6', 'ref_max_0_6', 'ref_max_2_6', 'ref_max_0_2_6']:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}")

display_df.columns = ['日期', 'H0价', 'H2价', 'H6价', 
                      'max(0,6)', 'max(2,6)', 'max(0,2,6)',
                      '周diff(0,6)vs(2,6)', '周diff(0,2,6)vs(0,6)', '周diff(0,2,6)vs(2,6)',
                      '累计(0,6)vs(2,6)', '累计(0,2,6)vs(0,6)', '累计(0,2,6)vs(2,6)']

display(display_df)

In [ ]:
# 保存结果
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
save_path = os.path.join(OUTPUT_DIR, f'strategy_comparison_3way_{timestamp}.xlsx')
df.to_excel(save_path, index=False)
print(f"Results saved: {save_path}")